# Domain Specific GPT with RAG and Open Source LLMs

## Introduction

In this lab exercise, we'll take a comprehensive look into **Retrieval Augmented Generation (RAG)** and concepts around memory for Large Language Models.

At a high-level, this lab will cover the following aspects:

1. Understand how Retrieval Augmented Generation (RAG) works at a high level and look into it in more detail
2. Explore Dense Passage Retrieval (DPR) and how it works
3. Compare RAG-Sequence and RAG-Token Models
4. Explore methods for long-term context and memory

## Setup and Dependencies

For this lab, we'll be using various packages for implementing RAG components. Make sure you have all required dependencies installed.

In [ ]:
# Import required libraries
import numpy as np
import torch
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

## Background: Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) is a hybrid approach for language generation as it utilizes **non-parametric memory** (samples not in training dataset) along with **parametric memory** (pretrained data) to generate responses utilizing the retrieved data without additional pretraining. 

### RAG Components

The approach is composed of two parts: the **retriever** and the **generator**.

**The Retriever:**
The retriever is composed of Dense Passage Retriever (DPR) which, in broad terms, indexes a set of passages into a low dimensional and continuous space to retrieve the most relevant passages from a provided input query.

**The Generator:**
The generator is a pretrained transformer which concatenates the input query $x$ with the retrieved content $z$ in order to generate a response using the augmented context.

### RAG Architecture Overview

For a query $x$:
- Maximum Inner Product Search (MIPS) finds top-K documents $z_i$
- For final prediction $y$, $z$ is treated as a latent variable
- The seq2seq predictions are marginalized to account for different documents

---
## Part 1: Dense Passage Retrieval

### Background

The Dense Passage Retriever (DPR) is the retrieval portion of RAG which utilizes two separate transformers to create the embeddings for the queries ($q$) and the embeddings for the passages ($p$). The embeddings for both the query and passage texts are obtained from the **class token** of the tokenized inputs. 

**Query Preprocessing:**
Query text is not chunked and is preprocessed by appending the character sequence with a separation token (`[SEP]`). For example, a preprocessed query would look something like:
```
What is machine learning? [SEP]
```

**Passage Preprocessing:**
Passage text is obtained by chunking source text, such as Wikipedia articles, into blocks consisting of **100 words** where the article title is prepended to the block and separated using a `[SEP]` token. For example:
```
Machine Learning [SEP] Machine learning (ML) is a field of study in artificial intelligence...
```

**Encoding Process:**
The tokenized output is obtained by first passing the input text through an encoder (e.g., BERT) network which generates the hidden states and appends a class token (`[CLS]`). The class token, which attends to all of the tokens via self-attention, provides the dense embedding used to determine similarity between the query and passage.

**Similarity Computation:**
The similarity is defined as the dot product of the encoders for the query and the passage vectors:

$$\text{sim}(q, p) = E_Q(q)^\top \cdot E_P(p)$$

where $E_Q$ is the query encoder and $E_P$ is the passage encoder.

---
## Question 2: Dense Passage Retrieval (Programming)

In this section, we'll implement a simplified version of Dense Passage Retrieval.

### Components to Implement:
1. `preprocess_query()` - Format queries with `[SEP]` token
2. `preprocess_passage()` - Combine title and text with `[SEP]` token
3. `chunk_document()` - Split documents into word chunks
4. `SimpleDPR.encode()` - Encode texts into dense embeddings
5. `SimpleDPR.index_passages()` - Build passage embedding index
6. `SimpleDPR.retrieve()` - Retrieve top-k passages using dot product similarity

In [ ]:
# DPR Preprocessing Functions

# Special token used in DPR
SEP_TOKEN = "[SEP]"

def preprocess_query(query: str) -> str:
    """
    Preprocess a query by appending the [SEP] token.
    
    In DPR, queries are formatted with a separation token at the end.
    
    Args:
        query: Input query string
    
    Returns:
        Formatted query with [SEP] token appended
        
    Example:
        >>> preprocess_query("What is machine learning?")
        "What is machine learning? [SEP]"
    """
    # Fill in: Return the query with [SEP] token appended
    # Hint: Use an f-string to combine query, a space, and SEP_TOKEN
    pass


def preprocess_passage(title: str, text: str) -> str:
    """
    Preprocess a passage by combining title and text with [SEP] token.
    
    In DPR, passages are formatted with the title prepended and separated
    from the content using a [SEP] token.
    
    Args:
        title: Passage title
        text: Passage content
    
    Returns:
        Formatted passage with title [SEP] text format
        
    Example:
        >>> preprocess_passage("Machine Learning", "ML is a field of study...")
        "Machine Learning [SEP] ML is a field of study..."
    """
    # Fill in: Return the title and text combined with [SEP] token
    # Hint: Format should be "title [SEP] text"
    pass


def chunk_document(title: str, text: str, chunk_size: int = 100) -> List[Dict[str, str]]:
    """
    Split a document into chunks of approximately chunk_size words.
    
    In DPR, long documents are split into smaller passages of approximately 
    100 words each. Each chunk retains the original title for context.
    
    Args:
        title: Document title
        text: Full document text
        chunk_size: Target number of words per chunk (default: 100)
    
    Returns:
        List of dictionaries, each containing 'title' and 'text' keys
        
    Example:
        >>> chunks = chunk_document("AI", "word " * 250, chunk_size=100)
        >>> len(chunks)
        3
    """
    # Fill in: Split the text into chunks of chunk_size words
    # Step 1: Split text into words
    # Step 2: Loop through words in increments of chunk_size
    # Step 3: Create a dictionary with "title" and "text" for each chunk
    # Step 4: Return the list of chunk dictionaries
    pass

In [ ]:
# SimpleDPR Class Implementation

from sentence_transformers import SentenceTransformer

class SimpleDPR:
    """
    A simplified Dense Passage Retriever.
    
    This class implements the core functionality of DPR:
    - Encoding queries and passages into dense embeddings
    - Building an index of passage embeddings
    - Retrieving top-k relevant passages for a query
    """
    
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        """
        Initialize the SimpleDPR retriever.
        
        Args:
            model_name: HuggingFace model name for the encoder
        """
        print(f"Loading model {model_name}...")
        self.model = SentenceTransformer(model_name, device="cpu")
        self.passage_embeddings: Optional[np.ndarray] = None
        self.passages: List[Dict[str, str]] = []
    
    def encode(self, texts: List[str]) -> np.ndarray:
        """
        Encode a list of texts into dense embeddings.
        
        Uses the sentence transformer model to generate embeddings.
        The embeddings come from the [CLS] token representation.
        
        Args:
            texts: List of text strings to encode
        
        Returns:
            Numpy array of shape (len(texts), embedding_dim)
        """
        # Fill in: Use self.model.encode() to encode the texts
        # Hint: Pass convert_to_numpy=True to get a numpy array
        pass
    
    def index_passages(self, passages: List[Dict[str, str]]) -> None:
        """
        Build an index of passage embeddings for retrieval.
        
        Preprocesses each passage (adding title [SEP] text format),
        encodes them into embeddings, and stores them for later retrieval.
        
        Args:
            passages: List of passage dictionaries, each containing:
                - "title": The passage title
                - "text": The passage content
        """
        # Fill in: Index the passages for later retrieval
        # Step 1: Store passages in self.passages
        # Step 2: Preprocess each passage using preprocess_passage()
        # Step 3: Encode all preprocessed passages using self.encode()
        # Step 4: Store embeddings in self.passage_embeddings
        pass
    
    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        """
        Retrieve the top-k most relevant passages for a query.
        
        Encodes the query, computes dot product similarity with all
        indexed passages, and returns the top-k results.
        
        Args:
            query: The search query string
            top_k: Number of passages to retrieve (default: 5)
        
        Returns:
            List of dictionaries, each containing:
                - "title": The passage title
                - "text": The passage content
                - "score": The similarity score (dot product)
        
        Raises:
            ValueError: If no passages have been indexed
        """
        if self.passage_embeddings is None or len(self.passages) == 0:
            raise ValueError("No passages indexed. Call index_passages first.")
        
        # Fill in: Retrieve top-k passages for the query
        # Step 1: Preprocess the query using preprocess_query()
        # Step 2: Encode the query using self.encode() (returns array, get first element)
        # Step 3: Compute dot product similarity: np.dot(self.passage_embeddings, query_embedding)
        # Step 4: Get top-k indices using np.argsort() (remember to reverse for descending order)
        # Step 5: Build and return results list with "title", "text", and "score" for each
        pass

In [ ]:
# Test DPR implementation

# Sample passages (simplified Wikipedia-style)
sample_passages = [
    {
        "title": "Machine Learning",
        "text": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves."
    },
    {
        "title": "Deep Learning",
        "text": "Deep learning is a type of machine learning based on artificial neural networks with multiple layers. These networks attempt to simulate the behavior of the human brain in processing data and creating patterns for decision making."
    },
    {
        "title": "Natural Language Processing",
        "text": "Natural language processing is a field of artificial intelligence that gives machines the ability to read, understand, and derive meaning from human languages. It combines computational linguistics with machine learning."
    },
    {
        "title": "Computer Vision",
        "text": "Computer vision is a field of artificial intelligence that trains computers to interpret and understand the visual world. Using digital images and deep learning models, machines can identify and classify objects."
    },
    {
        "title": "Reinforcement Learning",
        "text": "Reinforcement learning is an area of machine learning where an agent learns to make decisions by taking actions in an environment to maximize cumulative reward. It differs from supervised learning by not requiring labeled data."
    }
]

# Initialize DPR
dpr = SimpleDPR()

# Index passages
print("Indexing passages...")
dpr.index_passages(sample_passages)
print(f"Indexed {len(sample_passages)} passages\n")

# Test queries
queries = [
    "What is machine learning?",
    "How do neural networks work?",
    "What is NLP used for?"
]

for query in queries:
    print(f"Query: {query}")
    print("-" * 60)
    results = dpr.retrieve(query, top_k=3)
    for i, result in enumerate(results, 1):
        print(f"  {i}. [{result['title']}] (score: {result['score']:.4f})")
        print(f"     {result['text'][:80]}...")
    print()

---
## Question 3: RAG-Sequence and RAG-Token (Written)

### Background

The generation component $p_{\theta}(y_i \mid x, z, y_{1:i-1})$ of RAG utilizes an encoder-decoder model (originally BART-large) in order to generate responses from the query-passage inputs. 

Within their work, Lewis et al. offers a sequence model approach along with a token model approach for generating the output.

**RAG-Sequence Model:**
The RAG-Sequence model forms $k$ different generator model inputs for a given query. These inputs are sourced from individually top-K retrieved passages $z$ using the retriever $p_\eta(z|x)$ and couple together with the given query to produce $k$ different model output sequences where the highest scoring sequence is selected:

$$p_{\text{RAG-Sequence}}(y \mid x) \approx \sum_{z \in \text{top-k}(p(\cdot \mid x))} p_{\eta}(z \mid x) \prod_{i}^{N} p_{\theta}(y_i \mid x, z, y_{1:i-1})$$

**RAG-Token Model:**
The RAG-Token model applies the same concept but ranks individually generated tokens $y_{1:i-1}$ rather than the entire sequence, allowing for a mixture of different passages to influence the generation of the output sequence:

$$p_{\text{RAG-Token}}(y \mid x) \approx \prod_{i}^{N} \sum_{z \in \text{top-k}(p(\cdot \mid x))} p_{\eta}(z \mid x) p_{\theta}(y_i \mid x, z, y_{1:i-1})$$

**Key Finding:**
Of the two approaches, the authors found that the RAG-Token model outperformed the RAG-Sequence model in question answer tasks, specifically those in open-domain. This is attributed to the nature of open-domain questions being required to combine facts from multiple passages, and the token level approach allows for generation to switch passages mid-generation. In the broader scope, RAG was found to generate more specific and accurate results than the existing parametric-only seq2seq models available at the time.

---
## Question 4: RAG-Sequence and RAG-Token (Programming)

In this section, you will implement the probability computations for both RAG-Sequence and RAG-Token models.

**Key Insight:** The critical difference is **when** marginalization over passages occurs:
- **RAG-Sequence**: Sum over passages is *outside* the product over tokens
- **RAG-Token**: Sum over passages is *inside* the product over tokens

In [ ]:
# Utility function for numerical stability

def log_sum_exp(log_probs: np.ndarray) -> float:
    """
    Compute log(sum(exp(log_probs))) in a numerically stable way.
    
    Uses the log-sum-exp trick: log Σ exp(x_i) = max(x) + log Σ exp(x_i - max(x))
    
    Args:
        log_probs: Array of log probabilities
    
    Returns:
        Log of sum of exponentials
    """
    max_log_prob = np.max(log_probs)
    return max_log_prob + np.log(np.sum(np.exp(log_probs - max_log_prob)))

### Part 4.1: RAG-Sequence Implementation

Implement the RAG-Sequence probability computation.

In [ ]:
def compute_sequence_log_prob(token_log_probs: np.ndarray) -> float:
    """
    Compute log probability of a sequence: log(∏_i p(y_i|x,z,y_{1:i-1}))
    
    In RAG-Sequence, we need to compute the product of token probabilities.
    In log space, this becomes a sum of log probabilities.
    
    Args:
        token_log_probs: Array of log probabilities for each token [num_tokens]
    
    Returns:
        Log probability of the sequence
    """
    # Fill in: Sum the log probabilities to get sequence log probability
    # Hint: In log space, multiplication becomes addition
    pass


def marginalize_over_passages(passage_log_probs: np.ndarray, 
                               sequence_log_probs: np.ndarray) -> float:
    """
    Compute log(∑_z p(z|x) * p(y|x,z))
    
    This marginalizes the sequence probability over all retrieved passages.
    
    Args:
        passage_log_probs: Log retrieval probabilities [num_passages]
        sequence_log_probs: Log sequence probabilities for each passage [num_passages]
    
    Returns:
        Log marginalized probability
    """
    # Fill in: Implement marginalization over passages
    # Step 1: Combine passage and sequence log probs (add them)
    # Step 2: Use log_sum_exp for numerical stability
    # Hint: log p(z|x) + log p(y|x,z) = log(p(z|x) * p(y|x,z))
    pass


def rag_sequence_probability(passage_log_probs: np.ndarray,
                              token_log_probs_per_passage: np.ndarray) -> float:
    """
    Full RAG-Sequence computation.
    
    Implements: p_RAG-Sequence(y|x) ≈ Σ_z p(z|x) * ∏_i p(y_i|x,z,y_{1:i-1})
    
    Args:
        passage_log_probs: Log retrieval probabilities [num_passages]
        token_log_probs_per_passage: Array of shape (num_passages, seq_len)
                                     containing token log probs for each passage
    
    Returns:
        Log probability of output sequence
    """
    # Fill in: Implement the full RAG-Sequence probability computation
    # Step 1: For each passage, compute the sequence log probability
    #         using compute_sequence_log_prob
    # Step 2: Marginalize over passages using marginalize_over_passages
    pass

In [ ]:
# Test RAG-Sequence implementation

print("=== RAG-Sequence Demo ===")
print()

# Example: 3 passages, sequence length 4
passage_probs = np.array([0.5, 0.3, 0.2])  # Must sum to 1
passage_log_probs = np.log(passage_probs)

print(f"Passage probabilities: {passage_probs}")
print(f"Passage log probs: {passage_log_probs}")
print()

# Token-level generation probabilities for a 4-token sequence
token_probs_per_passage = np.array([
    [0.7, 0.6, 0.8, 0.5],  # Passage 1
    [0.5, 0.7, 0.6, 0.6],  # Passage 2
    [0.3, 0.4, 0.5, 0.4],  # Passage 3
])
token_log_probs_per_passage = np.log(token_probs_per_passage)

print("Token probabilities per passage:")
for i, probs in enumerate(token_probs_per_passage):
    print(f"  Passage {i+1}: {probs}")
print()

rag_seq_prob = rag_sequence_probability(passage_log_probs, token_log_probs_per_passage)
print(f"RAG-Sequence log probability: {rag_seq_prob:.4f}")
print(f"RAG-Sequence probability: {np.exp(rag_seq_prob):.6f}")

### Part 4.2: RAG-Token Implementation

Implement the RAG-Token probability computation.

In [ ]:
def marginalize_token_over_passages(passage_log_probs: np.ndarray,
                                     token_log_probs: np.ndarray) -> float:
    """
    Compute log(∑_z p(z|x) * p(y_i|x,z,y_{1:i-1})) for a single token position.
    
    This marginalizes a single token's probability over all passages.
    
    Args:
        passage_log_probs: Log retrieval probabilities [num_passages]
        token_log_probs: Log token probabilities for this position [num_passages]
    
    Returns:
        Log marginalized probability for this token
    """
    # Fill in: Marginalize this token's probability over all passages
    # Step 1: Combine passage and token log probs (add them)
    # Step 2: Use log_sum_exp to compute the marginalized probability
    # Hint: This is similar to marginalize_over_passages in rag_sequence.py
    #       but operates on a single token instead of a whole sequence
    pass


def rag_token_probability(passage_log_probs: np.ndarray,
                           token_log_probs_per_passage: np.ndarray) -> float:
    """
    Full RAG-Token computation.
    
    Implements: p_RAG-Token(y|x) ≈ ∏_i Σ_z p(z|x) * p(y_i|x,z,y_{1:i-1})
    
    Args:
        passage_log_probs: Log retrieval probabilities [num_passages]
        token_log_probs_per_passage: Array of shape (num_passages, seq_len)
                                      containing token log probs for each passage
    
    Returns:
        Log probability of output sequence
    """
    # Fill in: Implement the full RAG-Token probability computation
    # Step 1: Get the shape (k passages, seq_len tokens)
    # Step 2: For each token position i:
    #         a) Extract token log probs at position i for all passages
    #         b) Marginalize over passages using marginalize_token_over_passages
    # Step 3: Sum all marginalized log probs (product in prob space = sum in log space)
    # Hint: The key difference from RAG-Sequence is that marginalization happens
    #       PER TOKEN, not per sequence
    pass

In [ ]:
# Test RAG-Token implementation

print("=== RAG-Token Demo ===")
print()

# Use different example data to show passage switching
# Notice how different passages are better for different tokens
token_probs_per_passage = np.array([
    [0.8, 0.3, 0.7, 0.4],  # Passage 1: good at tokens 0, 2
    [0.3, 0.8, 0.4, 0.7],  # Passage 2: good at tokens 1, 3
    [0.4, 0.4, 0.5, 0.5],  # Passage 3: moderate overall
])
token_log_probs_per_passage = np.log(token_probs_per_passage)

print("Token probabilities per passage:")
for i, probs in enumerate(token_probs_per_passage):
    print(f"  Passage {i+1}: {probs}")
print()

rag_tok_prob = rag_token_probability(passage_log_probs, token_log_probs_per_passage)
print(f"RAG-Token log probability: {rag_tok_prob:.4f}")
print(f"RAG-Token probability: {np.exp(rag_tok_prob):.6f}")

print("\nComparison:")
print("  RAG-Token allows different passages to influence different tokens")
print("  RAG-Sequence commits to a single passage for the entire sequence")

---
## Question 5: Memory and Long-Term Context

### Background

Large language models are fundamentally constrained by their **context window** (maximum number of tokens they can process in a single forward pass). While recent models have expanded context windows significantly (from 4K to over 1M tokens), this constraint remains a fundamental bottleneck for tasks requiring persistent memory across long interactions or access to large knowledge bases.

**Two Primary Strategies:**

Two primary strategies have emerged to address this limitation:

1. **Long Context (LC):** Extending the model's context window to accommodate more tokens directly

2. **Retrieval-Augmented Generation (RAG):** Selectively retrieving relevant information from external storage when needed

**MemGPT and Hierarchical Memory:**

Recent work like MemGPT draws inspiration from operating systems, treating the context window as "main memory" and external storage as "disk," with the LLM managing data movement between tiers through function calls. This approach allows the model to:
- Keep the most relevant information in the limited "main context" (like RAM)
- Store other information in external storage (like disk)
- "Page in" relevant information when needed
- "Page out" less relevant information, similar to virtual memory in operating systems

---
## Summary

In this lab, we've explored:

1. **Dense Passage Retrieval (DPR)**
   - How queries and passages are preprocessed and encoded
   - The role of the `[CLS]` token in creating dense embeddings
   - Advantages of dense retrieval over sparse methods

2. **RAG-Sequence vs RAG-Token**
   - RAG-Sequence: Marginalizes over passages at the sequence level
   - RAG-Token: Marginalizes over passages at each token position
   - Trade-offs between single-source coherence and multi-source synthesis

3. **Memory and Long-Term Context**
   - Computational limitations of extending context windows (O(n²) complexity)
   - Trade-offs between Long Context and RAG approaches
   - Hierarchical memory systems inspired by operating systems

### Key Takeaways

- **RAG enables LLMs to access external knowledge** without retraining
- **Dense retrieval captures semantic similarity** beyond keyword matching
- **The choice between RAG-Sequence and RAG-Token** depends on whether coherence with a single source or multi-source synthesis is more important
- **Managing long-term memory** requires strategic approaches due to attention complexity constraints

---
## References

1. Lewis, P., et al. (2020). "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks." *NeurIPS 2020*.

2. Karpukhin, V., et al. (2020). "Dense Passage Retrieval for Open-Domain Question Answering." *EMNLP 2020*.

3. Devlin, J., et al. (2019). "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding." *NAACL 2019*.

4. Lewis, M., et al. (2019). "BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension." *ACL 2020*.

5. Packer, C., et al. (2024). "MemGPT: Towards LLMs as Operating Systems." *arXiv preprint*.

6. Vaswani, A., et al. (2017). "Attention Is All You Need." *NeurIPS 2017*.